# 前処理_毎勤
2026/3/17にe-Statからcsvファイルをダウンロード(表番号1)\
(https://www.e-stat.go.jp/stat-search/files?page=1&layout=datalist&toukei=00450071&tstat=000001011791&cycle=0&tclass1=000001035519&tclass2=000001144508&tclass3val=0)\
2025年12月までのデータが入っている

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_maikin = pd.read_csv('../data/raw/hon-maikin-k-jissu.csv', encoding='shift-jis')

## 格納データの確認

In [ ]:
df_maikin.head(3)

In [ ]:
df_maikin.tail(3)

In [ ]:
df_maikin.columns

In [ ]:
print('種別: ', df_maikin['種別'].unique())
print('年: ', df_maikin['年'].unique())
print('月: ', df_maikin['月'].unique())
print('産業分類: ', df_maikin['産業分類'].unique())
print('規模: ', df_maikin['規模'].unique())
print('就業形態: ', df_maikin['就業形態'].unique())

## 文字を置き換える
種別は実数のみ→消す\
月は不揃い→揃える\
産業分類・規模・就業形態は記号表記→実際の分類名に変換\
フィルターをかけて、調査産業計のみに絞る

In [ ]:
df_maikin['月'] = df_maikin['月'].replace({"CY": 99})
df_maikin['月'] = df_maikin['月'].astype(int)
df_maikin['産業分類'] = df_maikin['産業分類'].replace({
    'TL  ': '調査産業計',
    'C   ': '鉱業，採石業，砂利採取業',
    'D   ': '建設業',
    'E   ': '製造業',
    'F   ': '電気・ガス・熱供給・水道業',
    'G   ': '情報通信業',
    'H   ': '運輸業，郵便業',
    'I   ': '卸売業，小売業',
    'J   ': '金融業，保険業',
    'K   ': '不動産業，物品賃貸業',
    'L   ': '学術研究，専門・技術サービス業',
    'M   ': '宿泊業，飲食サービス業',
    'N   ': '生活関連サービス業，娯楽業',
    'O   ': '教育，学習支援業',
    'P   ': '医療，福祉',
    'Q   ': '複合サービス事業',
    'R   ': 'サービス業(他に分類されないもの)'
    })
df_maikin['規模'] = df_maikin['規模'].replace({
    '0': '30人以上',
    '4': '500人以上',
    '5': '100〜499人',
    '7': '30〜99人',
    '9': '5〜29人',
    'T': '5人以上'
    })
df_maikin['就業形態'] = df_maikin['就業形態'].replace({
    0: '形態計',
    1: '一般',
    2: 'パート'
    })

In [ ]:
df_maikin = df_maikin.filter(['年', '月', '産業分類', '規模', '就業形態', '現金給与総額', 'きまって支給する給与', '所定内給与',
       '所定外給与', '特別給与', '総実労働時間', '所定内労働時間', '所定外労働時間', '出勤日数', '前月末労働者数',
       '増加労働者数', '減少労働者数', '本月末労働者数', 'パートタイム労働者数'])
df_maikin = df_maikin.query('産業分類 == "調査産業計"').reset_index(drop=True)

## 2019年を基準に指数化
base_2019に2019年平均値のdfを作り、元のdfにマージ

In [ ]:
base_2019 = (
    df_maikin.query('年 == 2019 and 月 == 99')
    .filter(['産業分類', '規模', '就業形態', '現金給与総額', 'きまって支給する給与', '所定内給与', '所定外給与', '特別給与',
            '総実労働時間', '所定内労働時間', '所定外労働時間'])
    .rename(columns={
        '現金給与総額': '現金給与総額_2019',
        'きまって支給する給与': 'きまって支給する給与_2019',
        '所定内給与': '所定内給与_2019',
        '所定外給与': '所定外給与_2019',
        '特別給与': '特別給与_2019',
        '総実労働時間': '総実労働時間_2019',
        '所定内労働時間': '所定内労働時間_2019',
        '所定外労働時間': '所定外労働時間_2019'
        })
)

# 規模ごとに1行だけか確認
base_2019

In [ ]:
# assertは条件が真でないときerrorを返す
assert base_2019.duplicated(['産業分類', '規模', '就業形態']).sum() == 0

In [ ]:
df_maikin = df_maikin.merge(base_2019, on=['産業分類', '規模', '就業形態'],how='left')

df_maikin['現金給与総額指数'] = df_maikin['現金給与総額'] / df_maikin['現金給与総額_2019'] * 100
df_maikin['きまって支給する給与指数'] = df_maikin['きまって支給する給与'] / df_maikin['きまって支給する給与_2019'] * 100
df_maikin['所定内給与指数'] = df_maikin['所定内給与'] / df_maikin['所定内給与_2019'] * 100
df_maikin['所定外給与指数'] = df_maikin['所定外給与'] / df_maikin['所定外給与_2019'] * 100
df_maikin['特別給与指数'] = df_maikin['特別給与'] / df_maikin['特別給与_2019'] * 100
df_maikin['総実労働時間指数'] = df_maikin['総実労働時間'] / df_maikin['総実労働時間_2019'] * 100
df_maikin['所定内労働時間指数'] = df_maikin['所定内労働時間'] / df_maikin['所定内労働時間_2019'] * 100
df_maikin['所定外労働時間指数'] = df_maikin['所定外労働時間'] / df_maikin['所定外労働時間_2019'] * 100

## 毎勤の加工チェック

In [ ]:
df_maikin.head()

In [ ]:
df_maikin.columns

In [ ]:
print('年: ', df_maikin['年'].unique())
print('月: ', df_maikin['月'].unique())
print('産業分類: ', df_maikin['産業分類'].unique())
print('規模: ', df_maikin['規模'].unique())
print('就業形態: ', df_maikin['就業形態'].unique())

## いらない列を削除
産業分類は調査産業計のみなので削除\
XXXX_2019の列（8列）を削除\
出勤日数、増加労働者数、減少労働者数の3列を削除

In [ ]:
df_maikin = df_maikin.filter(['年', '月', '規模', '就業形態',
                                '現金給与総額', 'きまって支給する給与', '所定内給与', '所定外給与', '特別給与',
                                '総実労働時間', '所定内労働時間', '所定外労働時間',
                                '前月末労働者数', '本月末労働者数', 'パートタイム労働者数',
                                '現金給与総額指数', 'きまって支給する給与指数', '所定内給与指数', '所定外給与指数', '特別給与指数',
                                '総実労働時間指数', '所定内労働時間指数', '所定外労働時間指数'])

# 前処理_CPI
2026/3/17にe-Statからcsvファイルをダウンロード\
(https://www.e-stat.go.jp/stat-search/database?page=1&layout=datalist&toukei=00200573&tstat=000001150147&cycle=0&tclass1val=0)
- e-Stat上での操作
    - DBボタンを押下
    - 表示項目選択タブを選択
    - 表章項目は「指数」のみチェックを入れ、確定を押下
    - 2020年基準品目は「0163 持ち家の帰属家賃を除く総合」のみチェックを入れ、確定を押下
    - 地域は「全国」のみチェック入れ、確定を押下
    - 時間軸（年・月）はXXXX年度を除外する
        - どれでもいいので「XXXX年度」を選択
        - 同一階層の選択/解除で「解除」を押下
        - 確定を押下
    - 4項目の選定が終わったら、確定を押下
    - レイアウト設定タブを選択
    - ドラッグ＆ドロップで、「2020年基準品目」をページ上部（欄外）に、 時間軸（年・月）を行に移動
    - 設定して表示を更新を押下
    - 右上のダウンロードを押下
    - ファイル形式をCSV形式(列指向形式・Shift-JIS)に変更
    - ダウンロードを押下
    - 表ダウンロードのページに飛んだら、ダウンロードを押下
今回のファイル名はFEH_00200573_260317110412.csv

In [ ]:
df_cpi = pd.read_csv('../data/raw/FEH_00200573_260317110412.csv', encoding='shift-jis')

## 格納データの確認

In [ ]:
df_cpi.head(3)

In [ ]:
df_cpi.columns

In [ ]:
print('表章項目: ', df_cpi['表章項目'].unique())
print('2020年基準品目: ', df_cpi['2020年基準品目'].unique())
print('地域（2020年基準）: ', df_cpi['地域（2020年基準）'].unique())
print('時間軸（年・月）: ', df_cpi['時間軸（年・月）'].unique())

## データ前処理
使う列は時間軸（年・月）とvalueの2つ\
時間軸（年・月）はXXXX年XX月（各月の値）とXXXX年（年平均値）の2つが混在しているので、「年」列と「月」列に分ける

In [ ]:
df_cpi = df_cpi.filter(['時間軸（年・月）', 'value'])

#正規表現により年月分離
import re

def parse_year_month(value: str):
    """
    value: '2025年11月', '2025年' など
    return: (year, month)
    """

    # 年月（YYYY年MM月）
    m = re.fullmatch(r"(\d{4})年(\d{1,2})月", value)
    if m:
        return int(m.group(1)), int(m.group(2))

    # 年のみ（YYYY年）
    m = re.fullmatch(r"(\d{4})年", value)
    if m:
        return int(m.group(1)), 99

    # 想定外の形式
    return None, None


# 適用
df_cpi[["年", "月"]] = df_cpi["時間軸（年・月）"].apply(
    lambda x: pd.Series(parse_year_month(x))
)

In [ ]:
df_cpi = df_cpi.sort_values(["年", "月"]).reset_index(drop=True)\
     .filter(['年', '月', 'value'])\
    .rename(columns={'value': 'CPI(持ち家除く総合)'})

## CPIの加工チェック

In [ ]:
df_cpi.head(3)

# 2つのデータフレームを合体

In [ ]:
df = pd.merge(df_maikin, df_cpi, on=['年', '月'], how='left')

In [ ]:
df.head()

In [ ]:
df.columns

# 各種給与およびその指数を実質化

In [ ]:
df = df.assign(実質_現金給与総額指数 = df['現金給与総額指数']/df['CPI(持ち家除く総合)']*100,
            実質_きまって支給する給与指数 = df['きまって支給する給与指数']/df['CPI(持ち家除く総合)']*100,
            実質_所定内給与指数 = df['所定内給与指数']/df['CPI(持ち家除く総合)']*100,
            実質_所定外給与指数 = df['所定外給与指数']/df['CPI(持ち家除く総合)']*100,
            実質_特別給与指数 = df['特別給与指数']/df['CPI(持ち家除く総合)']*100,
            実質_現金給与総額 = df['現金給与総額']/df['CPI(持ち家除く総合)']*100,
            実質_きまって支給する給与 = df['きまって支給する給与']/df['CPI(持ち家除く総合)']*100,
            実質_所定内給与 = df['所定内給与']/df['CPI(持ち家除く総合)']*100,
            実質_所定外給与 = df['所定外給与']/df['CPI(持ち家除く総合)']*100,
            実質_特別給与 = df['特別給与']/df['CPI(持ち家除く総合)']*100)

In [ ]:
df.head()

# 処理したデータフレームをcsvで出力

In [ ]:
df.to_csv('../data/processed/毎勤CPI処理_調査産業計_20260317.csv', index=False, encoding='utf-8-sig')